# Does the machine lie? Measuring readout error

| | |
|---|---|
| **Level** | Introductory |
| **Time** | About 30 minutes |
| **Prerequisites** | Qubits, the X gate, measurement |
| **Default device** | Rigetti Cepheus-1-108Q |
| **Also runs on** | IQM Garnet, AQT IBEX Q1 |
| **Qubits** | 8 |
| **Two-qubit gates** | None |
| **Hardware jobs** | 1 |
| **Approximate cost** | Rigetti: about 10 credits (billed by execution time). Garnet at 1000 shots: about 175 credits. |
| **Suggested hand-in** | Your error-rate table and answers to Questions 1 and 2 |

The device, the number of shots, and whether to use hardware are set in the **Settings** cell below. Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.

*Part of the QUEST notebooks from qBraid. You may copy, edit and adapt this notebook for your course.*

When you measure a qubit, the result is supposed to tell you its state. On real hardware it sometimes does not: the device reports 1 when the qubit was in 0, or 0 when it was in 1. This is **readout error**.

In this notebook you prepare qubits in states you already know and count how often the hardware reports the wrong answer. The circuit uses only the X gate and measurement, so every mistake comes from the measurement itself, or from the qubit changing state just before it is read.

In [ ]:
# Settings. Change these, then run the notebook from the top.
DEVICE_ID = "rigetti:rigetti:qpu:cepheus-1-108q"   # device list and prices: see the README
SHOTS = 1000                  # measurements per hardware job
RUN_ON_HARDWARE = False     # set to True when you are ready to spend credits
QUEST_JOB_TAGS = {"quest": "found-readout"}   # labels this notebook's hardware jobs for QUEST usage statistics


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qbraid.runtime import QbraidProvider

In [ ]:
# Helper functions. You do not need to read these to follow the notebook.

def estimate_cost(device, n_jobs, shots):
    """Estimated qBraid credits for n_jobs hardware jobs (100 credits = $1)."""
    pricing = getattr(device.profile, "pricing", None)
    if pricing is None:
        return "no published price for this device"
    if float(pricing.perMinute) > 0:
        return ("billed by execution time, so it cannot be quoted in advance "
                f"({float(pricing.perMinute):,.0f} credits per minute of device time; "
                "small jobs cost about 10 credits each in our tests)")
    per_job = float(pricing.perTask) + float(pricing.perShot) * shots
    return f"about {n_jobs * per_job:,.0f} credits ({n_jobs} job(s) at {per_job:,.1f} each)"


def prob_one(counts, qubit, n_qubits):
    """Fraction of shots in which `qubit` was measured as 1. Qubit 0 is the rightmost bit."""
    total = sum(counts.values())
    ones = 0
    for key, n in counts.items():
        bits = key.replace(" ", "").zfill(n_qubits)
        if bits[-(qubit + 1)] == "1":
            ones += n
    return ones / total

## 1. Build the circuit

The circuit has eight qubits. Qubits 0 to 3 are left in $|0\rangle$. Qubits 4 to 7 get an X gate, which puts them in $|1\rangle$. Measuring all eight in one circuit costs one hardware job instead of eight.

In [ ]:
N_QUBITS = 8
prepared = [0, 0, 0, 0, 1, 1, 1, 1]   # the state each qubit is prepared in

qc = QuantumCircuit(N_QUBITS)
for qubit, bit in enumerate(prepared):
    if bit == 1:
        qc.x(qubit)
qc.measure_all()

qc.draw(output="text")

## 2. Ideal simulation

A perfect device reports the prepared state on every shot. Qiskit writes qubit 0 as the rightmost bit, so the expected result is `11110000`.

In [ ]:
simulator = AerSimulator()
ideal_counts = simulator.run(qc, shots=SHOTS).result().get_counts()
print(ideal_counts)

## 3. Run on hardware

The first cell connects to the device and prints the estimated cost. The second submits the job, if `RUN_ON_HARDWARE` is `True`.

In [ ]:
N_JOBS = 1

# Connect to the device and estimate the cost. This step is free.
try:
    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)
    status = device.status().name
    print(f"Device:         {DEVICE_ID}")
    print(f"Status:         {status}")
    print(f"Estimated cost: {estimate_cost(device, n_jobs=N_JOBS, shots=SHOTS)}")
    if status != "ONLINE":
        print("This device is not online right now. Some devices run in scheduled windows.")
        print("Try again later, or choose another device in Settings.")
except Exception as err:
    device = None
    print(f"Could not reach qBraid ({err}). The simulation sections still work.")

In [ ]:
if RUN_ON_HARDWARE and device is not None:
    job = device.run(qc, shots=SHOTS, tags=QUEST_JOB_TAGS)
    print("Submitted. Waiting for results; queues can take anywhere from seconds to hours.")
    hw_counts = job.result().data.get_counts()
    print(f"Received {sum(hw_counts.values())} shots.")
else:
    hw_counts = None
    print("Hardware step skipped. Set RUN_ON_HARDWARE = True in Settings to run it.")

## 4. Compare

For each qubit, the error rate is the fraction of shots in which the reported bit differs from the prepared bit.

In [ ]:
def error_rates(counts):
    rates = []
    for qubit, bit in enumerate(prepared):
        p1 = prob_one(counts, qubit, N_QUBITS)
        rates.append(p1 if bit == 0 else 1 - p1)
    return rates

ideal_err = error_rates(ideal_counts)
hw_err = error_rates(hw_counts) if hw_counts else None

print("qubit  prepared   ideal error   hardware error")
for q in range(N_QUBITS):
    hw = f"{hw_err[q]:.3f}" if hw_err else "   -"
    print(f"  {q}       {prepared[q]}          {ideal_err[q]:.3f}          {hw}")

if hw_err:
    colors = ["tab:blue" if b == 0 else "tab:orange" for b in prepared]
    plt.bar(range(N_QUBITS), hw_err, color=colors)
    plt.xlabel("qubit (blue: prepared in 0, orange: prepared in 1)")
    plt.ylabel("fraction of shots read wrong")
    plt.title(f"Readout error on {DEVICE_ID}")
    plt.show()

The ideal error is zero for every qubit. On hardware, expect a few percent per qubit, with noticeable differences from qubit to qubit.

Errors are often larger for qubits prepared in $|1\rangle$. A qubit in $|1\rangle$ can relax to $|0\rangle$ before the measurement finishes, while a qubit in $|0\rangle$ has no lower state to fall into.

## Questions to try

1. Which qubits had the largest error? Were the errors larger for qubits prepared in $|1\rangle$ or in $|0\rangle$?
2. Suppose each qubit has a 3% chance of being read wrong. If you measure 10 qubits, roughly what fraction of shots contains at least one wrong bit?
3. Change `DEVICE_ID` to IQM Garnet (`"aws:iqm:qpu:garnet"`) and run again. How do the two devices compare?
4. Set `SHOTS = 100` and run again. How much do the error rates change between runs? About how many shots do you need to measure a 2% effect reliably?

## Going further

Readout error can be measured and then partly undone. The qBraid Error-Mitigation series shows how, in its notebook *Readout Error Mitigation* (`tutorials/Error-Mitigation/02_readout_mitigation.ipynb` in this repository).